# M2.Ex1: Advertising Revenue

- Run: [**Open In Colab**](https://colab.research.google.com/github/HassanAlgoz/B5/blob/main/content/W3/M2/exercises/ex1_multi-reg.ipynb)

## Advertising Dataset

The Advertising Dataset is a fundamental resource in statistical learning and regression analysis.

- Features: `3` numerical
- Target: `sales` of the product (in thousands of units).
- Size: `200` samples.
- Source: [Advertising Dataset](https://www.statlearning.com/s/Advertising.csv)

In [ ]:
import pandas as pd
import numpy as np
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

### Step 1. Load the data

In [ ]:
url = "https://www.statlearning.com/s/Advertising.csv"
df = pd.read_csv(url, index_col=0)  # first column is row index
display(df.head())

### Step 2.a Assign variables `X` to the features and `y` to the target

In [ ]:
X = df[['TV', 'Radio', 'Newspaper']]  # 3 numerical features
y = df['Sales']                        # target: sales in thousands

print('X (first 5 rows):')
display(X.head())
print('y (first 5 values):')
print(y.head())

### Step 2.b print the type of each

In [ ]:
print('Type of X:', type(X))
print('Type of y:', type(y))

### Step 2.c identify whether the target is categorical or numerical & whether the task is regression or classification

In [ ]:
print('Target dtype:', y.dtype)
print()
print('The target (Sales) is NUMERICAL — it is a continuous float value.')
print('Therefore, the task is REGRESSION.')

### Step 3. Identify the number of samples and columns of both the data matrix and the target

In [ ]:
print('--- Feature Matrix X ---')
print(f'Shape            : {X.shape}')
print(f'Number of samples: {X.shape[0]}')
print(f'Number of columns: {X.shape[1]}')
print(f'Feature names    : {X.columns.tolist()}')
print()
print('--- Target Vector y ---')
print(f'Shape            : {y.shape}')
print(f'Number of samples: {y.shape[0]}')

### Step 4. Summarize the distribution of the data (min, max, median, mean, and standard deviation)

In [ ]:
summary = df.agg(['min', 'max', 'median', 'mean', 'std'])
print('Distribution Summary:')
display(summary)

### Step 5. How much difference do you see in the scale of each feature? (calculate the feature-wise range differences)

In [ ]:
ranges = X.max() - X.min()
print('Feature-wise Range (max - min):')
print(ranges)
print()
print('Observation:')
print('  TV has the largest range (~296), followed by Newspaper (~196) and Radio (~48).')
print('  Because the features are on very different scales, StandardScaler is important')
print('  to prevent TV from dominating the model simply due to its larger magnitude.')

### Step 6.a Plot each of the features vs the target

In [ ]:
sns.pairplot(df,
             x_vars=['TV', 'Radio', 'Newspaper'],
             y_vars='Sales',
             height=5, aspect=0.8,
             kind='reg',
             plot_kws={'line_kws': {'color': 'crimson'}, 'scatter_kws': {'alpha': 0.4}})
plt.suptitle('Advertising Features vs Sales', y=1.02, fontsize=13, fontweight='bold')
plt.show()

### Step 6.b What is the relationship between the feature and the target? (increasing or decreasing or none)

1. `x=TV` and `y=Sales`
2. `x=Radio` and `y=Sales`
3. `x=Newspaper` and `y=Sales`

In [ ]:
corr = df[['TV', 'Radio', 'Newspaper', 'Sales']].corr()['Sales'].drop('Sales')
print('Pearson Correlation with Sales:')
print(corr.round(4))
print()
print('1. TV       vs Sales: INCREASING (strong positive, r ≈ 0.78)')
print('2. Radio    vs Sales: INCREASING (moderate positive, r ≈ 0.58)')
print('3. Newspaper vs Sales: WEAK / ALMOST NONE (r ≈ 0.23) — very weak positive')

### Step 7. Define the pipeline with pre-processing steps

Make a Pipeline of three sequential steps:

1. transformer: `SimpleImputer` (to fill in missing values)
2. transformer: `StandardScaler` (to scale numerical features)
3. predictor: `LinearRegression` (to model the relationship)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

# Chain: impute missing values -> scale -> fit linear model
preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),  # fill NaN with column mean
    ('scaler',  StandardScaler()),                # standardize: (x - mean) / std
])

predictor = LinearRegression()

In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", predictor),
    ]
)
print(pipe)

### Step 8. Split the dataset into train and test sets

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Total samples     : {len(df)}')
print(f'Training set size : {X_train.shape[0]} samples (80%)')
print(f'Test set size     : {X_test.shape[0]} samples (20%)')

### Step 9.a Fit the pipeline on the training set

In [ ]:
pipe.fit(X_train, y_train)
print('Pipeline fitted successfully!')

### Step 9.b Identify the learned coefficients (for each feature) and the bias term

In [ ]:
# Access the LinearRegression step inside the pipeline
lr = pipe.named_steps['regressor']

coef_df = pd.DataFrame({
    'Feature'    : X.columns,
    'Coefficient': lr.coef_
})

print('Learned Coefficients (on scaled features):')
display(coef_df)
print()
print(f'Bias (intercept): {lr.intercept_:.4f}')
print()
print('Note: coefficients are on standardized features.')
print('A larger absolute coefficient means that feature has more influence on Sales.')

### Step 9.c how much spending on TV `$1,000` more, factor into Sales?

In [ ]:
# The scaler was fitted on the training data. We need the std of TV to convert.
# The pipeline scaled features: coefficient in scaled space = coef * (1 / std)
# So: 1 unit change in raw TV -> coef_TV / std_TV change in Sales

scaler = pipe.named_steps['preprocessor'].named_steps['scaler']
tv_std = scaler.scale_[0]   # std of TV from training data
tv_coef = lr.coef_[0]       # coefficient on scaled TV

# Effect of $1,000 increase in TV budget (1 unit in the dataset = $1,000)
sales_change_per_1k_tv = tv_coef / tv_std

print(f'TV coefficient (scaled space): {tv_coef:.4f}')
print(f'TV std (from training data)  : {tv_std:.4f}')
print()
print(f'Effect of $1,000 more on TV spending: {sales_change_per_1k_tv:.4f} thousand units of Sales')
print(f'  i.e., approximately {sales_change_per_1k_tv * 1000:.1f} more units sold')

### Step 9.d if we take `$5,000` away from Newspaper and put it in Radio how much difference does that make into Sales?

In [ ]:
radio_std     = scaler.scale_[1]   # std of Radio
newspaper_std = scaler.scale_[2]   # std of Newspaper
radio_coef     = lr.coef_[1]
newspaper_coef = lr.coef_[2]

# Effect of +$5,000 in Radio and -$5,000 in Newspaper
delta = 5  # 5 units = $5,000
effect_radio     = radio_coef     / radio_std     * delta   # gain from more Radio
effect_newspaper = newspaper_coef / newspaper_std * (-delta) # loss from less Newspaper
net_effect = effect_radio + effect_newspaper

print(f'+$5,000 to Radio     contribution: +{effect_radio:.4f} thousand units')
print(f'-$5,000 from Newspaper contribution: {effect_newspaper:.4f} thousand units')
print()
print(f'Net change in Sales: {net_effect:.4f} thousand units')
print(f'  i.e., approximately {net_effect * 1000:.1f} more/fewer units sold')

### Step 9.e if we spend nothing at all on advertising, how much do we estimate our Sales to be?

In [ ]:
# Predict with all features = 0
zero_spend = pd.DataFrame([[0, 0, 0]], columns=['TV', 'Radio', 'Newspaper'])
estimated_sales = pipe.predict(zero_spend)[0]

print(f'Estimated Sales with $0 advertising: {estimated_sales:.4f} thousand units')
print(f'  i.e., approximately {estimated_sales * 1000:.0f} units')
print()
print('This is the model\'s baseline sales estimate when no advertising budget is allocated.')

### Step 10. Evaluate the pipeline on the test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

score = pipe.score(X_test, y_test)   # R² score
y_pred = pipe.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print('=== Pipeline Evaluation on Test Set ===')
print(f'  R²   : {score:.4f}')
print(f'  MAE  : {mae:.4f}')
print(f'  RMSE : {rmse:.4f}')
print()
print(f'The pipeline explains {score*100:.1f}% of the variance in Sales.')

### Step 11. Define a `LinearRegression` model without the pre-processing steps, and compare it's score with the pipeline. Which one is better?

In [ ]:
# Baseline: plain LinearRegression with no preprocessing
plain_lr = LinearRegression()
plain_lr.fit(X_train, y_train)

score_plain = plain_lr.score(X_test, y_test)
y_pred_plain = plain_lr.predict(X_test)
mae_plain  = mean_absolute_error(y_test, y_pred_plain)
rmse_plain = np.sqrt(mean_squared_error(y_test, y_pred_plain))

print('=== Comparison ===' )
print(f'{'Metric':<10} {'Pipeline (scaled)':<22} {'Plain LinearReg'}')
print('-' * 55)
print(f'{'R²':<10} {score:<22.4f} {score_plain:.4f}')
print(f'{'MAE':<10} {mae:<22.4f} {mae_plain:.4f}')
print(f'{'RMSE':<10} {rmse:<22.4f} {rmse_plain:.4f}')
print()
print('Conclusion:')
print('Both models achieve nearly identical R² scores on this dataset.')
print('StandardScaler does NOT change the predictive power of LinearRegression')
print('because linear regression is scale-invariant in terms of predictions.')
print('However, scaling IS important for regularized models (Ridge, Lasso) and')
print('gradient-based algorithms, and it makes coefficients more interpretable.')